In [1]:
import os
import json
import shutil
from collections import Counter

# Dossier racine = dossier où se trouve ce notebook
ROOT        = os.getcwd()
IMAGES_ROOT = os.path.join(ROOT, 'BDD_100k')
LABELS_ROOT = os.path.join(ROOT, '100k')

print(f'Dossier racine : {ROOT}')
print(f'Images BDD_100k : {"OK" if os.path.exists(IMAGES_ROOT) else "INTROUVABLE"}')
print(f'Labels 100k     : {"OK" if os.path.exists(LABELS_ROOT) else "INTROUVABLE"}')

for split in ['train', 'val']:
    img_dir = os.path.join(IMAGES_ROOT, split)
    lbl_dir = os.path.join(LABELS_ROOT, split)
    n_img = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')]) if os.path.exists(img_dir) else 0
    n_lbl = len([f for f in os.listdir(lbl_dir) if f.endswith('.json')]) if os.path.exists(lbl_dir) else 0
    print(f'{split.upper()} : {n_img} images | {n_lbl} labels JSON')

Dossier racine : C:\Users\LENOVO\Desktop\Projet_Intelligence_Artificiell
Images BDD_100k : OK
Labels 100k     : OK
TRAIN : 70000 images | 70000 labels JSON
VAL : 10000 images | 10000 labels JSON


In [2]:

def filter_split(split):
    lbl_dir    = os.path.join(LABELS_ROOT, split)
    json_files = [f for f in os.listdir(lbl_dir) if f.endswith('.json')]
    highway    = []
    scenes     = Counter()

    print(f'Lecture {split} : {len(json_files)} fichiers JSON...')

    for i, jf in enumerate(json_files):
        with open(os.path.join(lbl_dir, jf), 'r') as f:
            data = json.load(f)

        scene = data.get('attributes', {}).get('scene', 'unknown')
        scenes[scene] += 1

        if scene == 'highway':
            highway.append(jf)

        if (i + 1) % 5000 == 0:
            print(f'  Progression : {i+1}/{len(json_files)}...')

    print(f'\nDistribution scènes ({split}) :')
    for s, c in scenes.most_common():
        print(f'  {s:<25} : {c:>6}{" <- NOTRE SCENARIO" if s == "highway" else ""}')

    return highway


print('=== FILTRAGE TRAIN ===')
train_hw = filter_split('train')

print('\n=== FILTRAGE VAL ===')
val_hw = filter_split('val')

print(f'\n--- RÉSULTAT ---')
print(f'Train autoroute : {len(train_hw)} images')
print(f'Val autoroute   : {len(val_hw)} images')
print(f'TOTAL           : {len(train_hw) + len(val_hw)} images')

=== FILTRAGE TRAIN ===
Lecture train : 70000 fichiers JSON...
  Progression : 5000/70000...
  Progression : 10000/70000...
  Progression : 15000/70000...
  Progression : 20000/70000...
  Progression : 25000/70000...
  Progression : 30000/70000...
  Progression : 35000/70000...
  Progression : 40000/70000...
  Progression : 45000/70000...
  Progression : 50000/70000...
  Progression : 55000/70000...
  Progression : 60000/70000...
  Progression : 65000/70000...
  Progression : 70000/70000...

Distribution scènes (train) :
  city street               :  43581
  highway                   :  17414 <- NOTRE SCENARIO
  residential               :   8105
  parking lot               :    378
  undefined                 :    366
  tunnel                    :    129
  gas stations              :     27

=== FILTRAGE VAL ===
Lecture val : 10000 fichiers JSON...
  Progression : 5000/10000...
  Progression : 10000/10000...

Distribution scènes (val) :
  city street               :   6112
  highway  

In [3]:
OUTPUT      = os.path.join(ROOT, 'bdd100k_highway')
OUT_TR_IMG  = os.path.join(OUTPUT, 'images', 'train')
OUT_VAL_IMG = os.path.join(OUTPUT, 'images', 'val')
OUT_TR_LBL  = os.path.join(OUTPUT, 'labels', 'train')
OUT_VAL_LBL = os.path.join(OUTPUT, 'labels', 'val')

for folder in [OUT_TR_IMG, OUT_VAL_IMG, OUT_TR_LBL, OUT_VAL_LBL]:
    os.makedirs(folder, exist_ok=True)


def copy_highway(hw_files, split, out_img, out_lbl):
    img_dir = os.path.join(IMAGES_ROOT, split)
    lbl_dir = os.path.join(LABELS_ROOT, split)
    copied, missing = 0, 0

    for jf in hw_files:
        img_name = jf.replace('.json', '.jpg')
        src_img  = os.path.join(img_dir, img_name)
        src_lbl  = os.path.join(lbl_dir, jf)

        if not os.path.exists(src_img):
            missing += 1
            continue

        shutil.copy2(src_img, os.path.join(out_img, img_name))
        shutil.copy2(src_lbl, os.path.join(out_lbl, jf))
        copied += 1

        if copied % 500 == 0:
            print(f'  {split} : {copied}/{len(hw_files)} copiées...')

    print(f'{split.upper()} — Copiées : {copied} | Manquantes : {missing}')
    return copied


print('=== Copie TRAIN ===')
n_train = copy_highway(train_hw, 'train', OUT_TR_IMG, OUT_TR_LBL)

print('\n=== Copie VAL ===')
n_val = copy_highway(val_hw, 'val', OUT_VAL_IMG, OUT_VAL_LBL)

print(f'\n=== TERMINÉ ===')
print(f'Train : {n_train} images')
print(f'Val   : {n_val} images')
print(f'\nDossier créé : {OUTPUT}')
print('Prochaine étape : uploader bdd100k_highway/ sur Roboflow')

=== Copie TRAIN ===
  train : 500/17414 copiées...
  train : 1000/17414 copiées...
  train : 1500/17414 copiées...
  train : 2000/17414 copiées...
  train : 2500/17414 copiées...
  train : 3000/17414 copiées...
  train : 3500/17414 copiées...
  train : 4000/17414 copiées...
  train : 4500/17414 copiées...
  train : 5000/17414 copiées...
  train : 5500/17414 copiées...
  train : 6000/17414 copiées...
  train : 6500/17414 copiées...
  train : 7000/17414 copiées...
  train : 7500/17414 copiées...
  train : 8000/17414 copiées...
  train : 8500/17414 copiées...
  train : 9000/17414 copiées...
  train : 9500/17414 copiées...
  train : 10000/17414 copiées...
  train : 10500/17414 copiées...
  train : 11000/17414 copiées...
  train : 11500/17414 copiées...
  train : 12000/17414 copiées...
  train : 12500/17414 copiées...
  train : 13000/17414 copiées...
  train : 13500/17414 copiées...
  train : 14000/17414 copiées...
  train : 14500/17414 copiées...
  train : 15000/17414 copiées...
  train : 